# 04 · NLP Preprocessing

MailMind uses a **security-preserving** NLP preprocessing strategy:
- Standard text normalization for the TF-IDF classifier
- **Separate** extraction of security-sensitive indicators (URLs, domains, phone numbers, currency symbols)

This dual-track approach lets the ML model learn from clean text while the feature-engineering layer retains raw security signals.

In [ ]:
import sys
import re
import html
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup

sys.path.insert(0, str(Path('../backend').resolve()))
from ml_pipeline import clean_nlp_text, extract_features

df = pd.read_csv('../data/processed/dataset.csv')
print(f'Dataset: {len(df):,} messages')

## 1. Text Cleaning Pipeline

The `clean_nlp_text()` function performs:
1. HTML tag removal via BeautifulSoup
2. HTML entity decoding
3. Whitespace normalization
4. Lowercasing

In [ ]:
# Demonstrate the cleaning function
examples = [
    '<b>URGENT:</b> Click <a href="http://malicious.test">HERE</a> to verify your account!!!',
    'Hi John, the meeting is at 3pm tomorrow. Best regards,\nAlice',
    'FREE!!! Win £1000 cash NOW. Text WIN to 80800. T&Cs apply.',
]

for text in examples:
    cleaned = clean_nlp_text(text)
    print(f'Raw:     {text[:80]}')
    print(f'Cleaned: {cleaned[:80]}')
    print()

## 2. Apply Cleaning to Full Dataset

In [ ]:
df['cleaned'] = df['text'].apply(clean_nlp_text)

# Compare lengths
df['raw_len'] = df['text'].str.len()
df['clean_len'] = df['cleaned'].str.len()
df['len_diff'] = df['raw_len'] - df['clean_len']

print('Character count changes after cleaning:')
print(df[['raw_len', 'clean_len', 'len_diff']].describe().round(1))

# Verify no empty results
empty_after = (df['cleaned'].str.strip() == '').sum()
print(f'\nMessages empty after cleaning: {empty_after}')

## 3. Token Analysis Post-Cleaning

In [ ]:
df['token_count'] = df['cleaned'].str.split().str.len()

print('Token count by class after cleaning:')
print(df.groupby('label')['token_count'].describe().round(1))

# Vocabulary size estimate
all_tokens = set()
for text in df['cleaned']:
    all_tokens.update(text.split())
print(f'\nUnique tokens (after cleaning): {len(all_tokens):,}')

## 4. Security Signal Extraction (Separate Track)

While the TF-IDF model operates on cleaned text, the feature-engineering layer extracts raw security signals separately. This avoids losing critical indicators during normalization.

In [ ]:
# Demonstrate feature extraction on sample messages
test_cases = [
    ('spammer@win.com', 'YOU WON!!!', 'Click http://bit.ly/prize to claim $5000 USD now! Call 1-800-555-0199'),
    ('alice@company.com', 'Meeting tomorrow', 'Hi, can we meet at 2pm to discuss the Q3 budget?'),
]

for sender, subject, body in test_cases:
    features = extract_features(sender, subject, body)
    print(f'--- {sender} | {subject} ---')
    for key in ['urlCount', 'suspiciousTermCount', 'shortenedUrlCount', 'uppercaseRatio', 'currencyCount', 'exclamationCount']:
        print(f'  {key}: {features[key]}')
    print()

In [ ]:
print('✅ NLP preprocessing pipeline documented and verified.')
print('Key design decisions:')
print('  1. Dual-track: clean text for ML + raw signals for security features')
print('  2. HTML stripped for TF-IDF but HTML element count preserved as a feature')
print('  3. Case normalized for ML but uppercase ratio preserved as a feature')
print('  4. URLs removed from clean text but counted and inspected separately')